# Parcial (2da parte)

Pregunta

Crea un agente que juegue captura de gemas evitando enemigos simples.

## El juego

El agente simulará el juego de búsqueda de gemas, en este el agente, tratará de capturar las gemas usando la estrategia con menores movimientos, que optimice las estrategias del juego, el ambiente tratará de un tablero de dimensiones de n*m, y en los epacios pondrá de manera aleatoria un determinado número de gemas, el tablero actuara como un array, las gemas tenfrán una ubicacion de tipo (x,y), al inicio de cada juego el agente empezará en la posición (0,0), mientras más gemas atrape mejor la recopensa que obtendrá, y penalizacion si es que toca al enemigo en su ubicación de (x,y), ls enemigos serán estáticos, no se moverán, y serán ubicados de manera aleatoria dentro de las dimensiones de mi tablero, el estado tras cada movimiento, captará donde se halla el agente, cuantas gemas le falta recolectar y donde se encuentran los enemigos.
El movimiento del agente dependerá de 4 estados [0, 1, 2, 3], que le permitirá moverse arriba, abajo, izquierda y derecha, resectivamente
El juego termina si es que el agente toca a un enemigo, el cual será estático, o cuando cupla un determinado número de pasos, debido a que los enemigos son estáticos, pueden encerrar a una gema, por lo que el límite de pasos me asegura que el agente dejará de jugar en un punto, además que cada paso dado penaliza al agente, por lo que siempre buscará las rutas con movimientos óptimos, tambien se termina el juego si el número de gemas llega a 0

In [6]:
import numpy as np
from tqdm import tqdm

class GemaEnv():

    def __init__(self, n=5, m=5, num_gemas=3,
                 num_enemigos=2, max_pasos=60):

        self.n = n
        self.m = m
        self.num_gemas = num_gemas
        self.num_enemigos = num_enemigos
        self.max_pasos = max_pasos

        self.actions = [0, 1, 2, 3]

    def reset(self):

        self.agente = [0, 0]
        self.pasos = 0

        posiciones = [
            (i, j)
            for i in range(self.n)
            for j in range(self.m)
            if (i, j) != (0, 0)
        ]

        np.random.shuffle(posiciones)

        self.gemas = set(posiciones[:self.num_gemas])

        self.enemigos = [
            list(posiciones[self.num_gemas + i])
            for i in range(self.num_enemigos)
        ]

        return self._get_state()

    def _get_state(self):

        return (
            tuple(self.agente),
            frozenset(self.gemas),
            tuple(tuple(e) for e in self.enemigos)
        )

    def _mover_enemigos(self):
        pass

    def step(self, action):

        self.pasos += 1

        if action == 0:
            self.agente[0] = max(0, self.agente[0] - 1)

        elif action == 1:
            self.agente[0] = min(self.n - 1, self.agente[0] + 1)

        elif action == 2:
            self.agente[1] = max(0, self.agente[1] - 1)

        elif action == 3:
            self.agente[1] = min(self.m - 1, self.agente[1] + 1)

        recompensa = -0.01

        pos = tuple(self.agente)

        if pos in self.gemas:
            self.gemas.remove(pos)
            recompensa += 10

        self._mover_enemigos()

        for enemigo in self.enemigos:

            if self.agente == enemigo:
                return self._get_state(), -50, True

        if len(self.gemas) == 0:
            return self._get_state(), 50, True

        if self.pasos >= self.max_pasos:
            return self._get_state(), -20, True

        return self._get_state(), recompensa, False


El agente está usando las funciones de gradiente, eso significa qeu a diferencia de otros ejemplos como el tres en raya, no está tratando de recordar posiciones o estados y ver cuál es la mejor opcion en una Q-table , sino que en base a las recompensas y a las penalizaciones, creará preferencias en sus movimientos, en las partidas irá explorando cuanto ueda y en base a lo que haya hecho, ve´ra cuanta recompensa o falta de esta le han dado, si las recompensasfueron altas, tomará los movimientos ue haya hecho como preferentes y en base a ellos creará su propia tabla de preferencias con ayuda de la función de softmax que convetirá en probabilidades los resultados de sus exploraciones, es decir que tan probable es que use movimientos iguales o similares que antes ya lo hallan llevado a una buena recompensa

In [7]:
class AgentGradiente:

    def __init__(self, alpha=0.01):

        self.preferences = {}
        self.alpha = alpha

        self.states = []
        self.actions = []

        self.recompensas = []

    def reset(self):

        self.states = []
        self.actions = []

    def get_preferences(self, state):

        if state not in self.preferences:
            self.preferences[state] = np.zeros(4)

        return self.preferences[state]

    def softmax(self, H):

        H = np.array(H)

        exp_H = np.exp(H - np.max(H))

        return exp_H / np.sum(exp_H)

    def move(self, env, state, explore=True):

        H = self.get_preferences(state)

        pi = self.softmax(H)

        if explore:
            action = np.random.choice(env.actions, p=pi)
        else:
            action = np.argmax(pi)

        self.states.append(state)
        self.actions.append((action, pi))

        return action

    def reward(self, reward):

        self.recompensas.append(reward)

        recompensa_media = np.mean(self.recompensas)

        for state, (action, pi) in zip(self.states, self.actions):

            H = self.get_preferences(state)

            for a in range(len(pi)):

                if a == action:

                    H[a] += self.alpha * (
                        reward - recompensa_media
                    ) * (1 - pi[a])

                else:

                    H[a] -= self.alpha * (
                        reward - recompensa_media
                    ) * pi[a]

            self.preferences[state] = H

Finalmente se entrenará al modelo en base a las reglas anteriormente establecidas, con 100000 episodios

In [9]:
class Trainer:

    def __init__(self, env, agent):

        self.env = env
        self.agent = agent

    def train(self, episodios=100000):

        recompensas = []

        for episodio in tqdm(range(episodios)):

            state = self.env.reset()

            self.agent.reset()

            done = False

            recompensa_total = 0

            while not done:

                action = self.agent.move(
                    self.env,
                    state,
                    explore=True
                )

                next_state, reward, done = self.env.step(action)

                recompensa_total += reward

                state = next_state

            self.agent.reward(recompensa_total)

            recompensas.append(recompensa_total)

            if episodio % 1000 == 0:

                promedio = np.mean(recompensas[-1000:])

                print(
                    f"Episodio: {episodio} | "
                    f"Promedio: {promedio:.2f}"
                )

        return recompensas

    def play(self):

        state = self.env.reset()

        done = False

        recompensa_total = 0

        print("Agente:", self.env.agente)
        print("Gemas:", self.env.gemas)
        print("Enemigos:", self.env.enemigos)
        print("----------------")

        while not done:

            action = self.agent.move(
                self.env,
                state,
                explore=False
            )

            state, reward, done = self.env.step(action)

            recompensa_total += reward

            print("Accion:", action)
            print("Agente:", self.env.agente)
            print("Gemas:", self.env.gemas)
            print("----------------")

        print("Recompensa final:", recompensa_total)

Entrenamiento con entorno establecido

In [11]:
env = GemaEnv(
    n=5,
    m=5,
    num_gemas=3,
    num_enemigos=2,
    max_pasos=60
)

agent = AgentGradiente(alpha=0.01)

trainer = Trainer(env, agent)

trainer.train(100000)

trainer.play()

  0%|          | 96/100000 [00:00<01:45, 950.39it/s]

Episodio: 0 | Promedio: -40.40


  1%|          | 1167/100000 [00:01<01:54, 866.57it/s]

Episodio: 1000 | Promedio: -30.73


  2%|▏         | 2147/100000 [00:02<01:47, 911.42it/s]

...

Episodio: 98000 | Promedio: -26.68


 99%|█████████▉| 99016/100000 [05:45<00:07, 135.53it/s]

Episodio: 99000 | Promedio: -27.72


100%|██████████| 100000/100000 [05:51<00:00, 284.29it/s]

Agente: [0, 0]
Gemas: {(1, 0), (4, 4), (0, 1)}
Enemigos: [[2, 2], [2, 0]]
----------------
Accion: 0
Agente: [0, 0]
Gemas: {(1, 0), (4, 4), (0, 1)}
----------------
Accion: 0
Agente: [0, 0]
Gemas: {(1, 0), (4, 4), (0, 1)}
----------------
Accion: 0
Agente: [0, 0]
Gemas: {(1, 0), (4, 4), (0, 1)}
----------------
Accion: 0
Agente: [0, 0]
Gemas: {(1, 0), (4, 4), (0, 1)}
----------------
Accion: 0
Agente: [0, 0]
Gemas: {(1, 0), (4, 4), (0, 1)}
----------------
Accion: 0
Agente: [0, 0]
Gemas: {(1, 0), (4, 4), (0, 1)}
----------------
Accion: 0
Agente: [0, 0]
Gemas: {(1, 0), (4, 4), (0, 1)}
----------------
Accion: 0
Agente: [0, 0]
Gemas: {(1, 0), (4, 4), (0, 1)}
----------------
Accion: 0
Agente: [0, 0]
Gemas: {(1, 0), (4, 4), (0, 1)}
----------------
Accion: 0
Agente: [0, 0]
Gemas: {(1, 0), (4, 4), (0, 1)}
----------------
Accion: 0
Agente: [0, 0]
Gemas: {(1, 0), (4, 4), (0, 1)}
----------------
Accion: 0
Agente: [0, 0]
Gemas: {(1, 0), (4, 4), (0, 1)}
----------------
Accion: 0
Agente: [0,

El entrenamiento del agente y las recompensas al inicio parecen ser malas, porque explota todos sus pasos o se choca con los enemigos, mientras más pasa mejora su entendimiento del juego, empieza a ganarlo y de a poco disminuye los pasos que usará, todo esto basado en el tamaño etablecido del tablero, que para las pruebas fue 5*5, 3 gemas que obtener, y 2 enemigos que evitar, otro factor influyente es el numero de pasos máximos que puede usar, que puede estar influyendo enlas bajas recompensas que recibe, y después guardamos las preferencias que tuvo en el entrenamiento

In [12]:
import pickle

with open('agente_gradiente.pickle', 'wb') as handle:
    pickle.dump(agent.preferences, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [13]:
with open('agente_gradiente.pickle', 'rb') as handle:

    preferencias = pickle.load(handle)

agent.preferences = preferencias